# Day 2: Making decisions and writing functions

**Time:** about 1 hour  |  **Needs:** Day 1
**Reservoir idea:** how small should the time step be? Decide with a rule, and package the calculation so you can reuse it
**Python you will learn:** True/False values, comparisons, `if` / `elif` / `else`, `try` / `except`, built-in functions, writing your own functions (`def`, parameters, `return`), looping over a list of values

By the end of today you will be able to:

1. write conditions such as `water_cut > 0.9` and combine them with `and`, `or`, `not`,
2. make a program choose between actions with `if` / `elif` / `else`,
3. protect a calculation against bad input with `try` / `except`,
4. write a function that takes inputs and returns a result, and tell the difference between `print` and `return`,
5. use a function to run the Day 1 time-stepping model for many time steps in one loop.

> **Companion reading (optional, about 45 minutes).** Charles Severance, *Python for Everybody* (free online: <https://www.py4e.com/html3/>):
> - Chapter 3 "Conditionals": "Boolean expressions", "Logical operators", "Conditional execution", "Alternative execution", "Chained conditionals", "Nested conditionals", "Catching exceptions using try and except"
> - Chapter 4 "Functions": "Function calls", "Built-in functions", "Type conversion functions", "Math functions", "Adding new functions", "Definitions and uses", "Flow of execution", "Parameters and arguments", "Fruitful functions and void functions", "Why functions?"
>


## 0. Setup and recap

Run the next cell. It loads the tools and re-creates the bacteria numbers from Day 1 ($\alpha=1.74$ per hour, $N_0=100$), so today's notebook stands on its own.

**Recap of Day 1.** Equation (2): $N(t)=N_0e^{\alpha t}$ is exact. Equation (4): $N^{n+1}=(1+\alpha\Delta t)N^n$ is the numerical recipe, and its error shrinks as $\Delta t$ shrinks. Today we ask: *which $\Delta t$ is small enough?* To answer that in code, we need decisions and functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

alpha = 1.74      # growth constant, per hour
N0 = 100          # starting number of bacteria

print("ready")

## 1. Story: the well that was shut in by a rule

*(An invented but typical situation.)* A field has a written operating rule: *"If the water cut of a well goes above 90 %, shut it in."* A new engineer is asked to automate the check over 200 wells every morning. Nobody wants to read 200 numbers by eye, so the check is written as a program: it looks at each well, asks a yes/no question, and does something different depending on the answer.

That is what today is about: asking yes/no questions in Python, and wrapping a set of steps into a named, reusable **function** so you can write the rule once and apply it 200 times.

## 2. True and False: comparisons

A comparison asks a yes/no question and gives back a value called a **boolean**: either `True` or `False`.

| Question | Python | Example |
|---|---|---|
| equal to? | `==` | `water_cut == 0.9` |
| not equal to? | `!=` | `water_cut != 0.9` |
| greater than? | `>` | `water_cut > 0.9` |
| less than? | `<` | `water_cut < 0.9` |
| greater than or equal? | `>=` | `water_cut >= 0.9` |
| less than or equal? | `<=` | `water_cut <= 0.9` |

> **Careful:** a single `=` *stores* a value (`x = 5`). Two equals signs `==` *ask a question* (`x == 5`). Mixing them up is the most common beginner error.

Predict what each line prints, then run the cell.

In [ ]:
water_cut = 0.93

print(water_cut > 0.9)
print(water_cut == 0.9)
print(water_cut != 0.9)
print(type(water_cut > 0.9))

### Combining questions: `and`, `or`, `not`

- `A and B` is `True` only if **both** are true.
- `A or B` is `True` if **at least one** is true.
- `not A` flips `True` to `False` and back.

```python
water_cut = 0.93
pressure = 1900                                   # psi
shut_in = water_cut > 0.9 or pressure < 1000      # either problem is enough
```

### A surprise about decimals

Run this and read the answer carefully:

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

Computers store decimals in binary, and most decimals such as 0.1 cannot be stored exactly. The tiny leftover can make `==` between two decimal results say `False`. To compare decimals, ask *"are they close?"* instead:

```python
abs(a - b) < 1e-9        # abs() gives the size of the difference
```
(`1e-9` is scientific notation for 0.000000001.) This is the reason Day 1 used `round(t_end / dt)` rather than comparing times with `==`.

**Your turn 2.1.** A grid block has water saturation `sw = 0.22`. The connate (irreducible) water saturation is `swc = 0.25` and the residual oil saturation is `sor = 0.20`. Water can only move if `sw` is above `swc`; oil can only move if the oil saturation `1 - sw` is above `sor`. Create three variables:

- `water_mobile`: is the water mobile?
- `oil_mobile`: is the oil mobile?
- `both_mobile`: are both mobile? (use `and`)

In [ ]:
sw = 0.22
swc = 0.25
sor = 0.20

# YOUR CODE HERE
water_mobile = None
oil_mobile = None
both_mobile = None

In [ ]:
# Check (do not edit)
assert water_mobile is False, f"water_mobile should be False (0.22 is below 0.25), got {water_mobile}"
assert oil_mobile is True, f"oil_mobile should be True (1 - 0.22 = 0.78 is above 0.20), got {oil_mobile}"
assert both_mobile is False, f"both_mobile should be False, got {both_mobile}"
print("Correct")

<details><summary>Solution</summary>

```python
water_mobile = sw > swc
oil_mobile = (1 - sw) > sor
both_mobile = water_mobile and oil_mobile
```
</details>

## 3. Making a choice: `if`, `else`, `elif`

```python
if condition:
    # lines here run only when the condition is True
else:
    # lines here run otherwise
```

Three details matter, exactly as for loops on Day 1: the **colon** at the end of the `if` line, the **indentation** (4 spaces) of the lines that belong to it, and that `else` lines up with `if`.

Here is the shut-in rule from the story. Predict: what happens if `water_cut` is exactly 0.90?

In [ ]:
water_cut = 0.93

if water_cut > 0.9:
    print("shut the well in")
else:
    print("keep producing")

Try `water_cut = 0.90` and `0.85` and run again. With exactly 0.90, the question `0.90 > 0.9` is `False`, so the well keeps producing. If the rule was meant to be "0.9 or more", you would write `>=`. Small choices like this matter in real engineering rules, so always test the *boundary*.

### More than two choices: `elif`

`elif` (short for "else if") adds more questions. Python tries them from top to bottom and runs only the **first** one that is true.

```python
if water_cut < 0.1:
    print("dry")
elif water_cut < 0.5:
    print("moderate")
else:
    print("high")
```

**Your turn 3.1.** A well *demands* `demand = 3200` STB/day, but the surface facilities allow at most `limit = 2500` STB/day. Using `if` / `else`, store in `rate` what the well will actually produce: the limit if the demand is above it, otherwise the demand.

In [ ]:
demand = 3200
limit = 2500

# YOUR CODE HERE
rate = None

In [ ]:
# Check (do not edit)
assert rate == 2500, f"rate should be 2500 (demand 3200 is above the limit), got {rate}"
print("Correct. Now change demand to 1800 above and run both cells again: the rate should follow the demand.")

<details><summary>Solution</summary>

```python
if demand > limit:
    rate = limit
else:
    rate = demand
```
</details>

**Your turn 3.2.** Classify a well's water cut `fw = 0.62` into a class, stored as text in `fw_class`, using `if` / `elif` / `else`:

| water cut | class |
|---|---|
| below 0.1 | `"dry"` |
| 0.1 up to (not including) 0.5 | `"moderate"` |
| 0.5 up to (not including) 0.9 | `"high"` |
| 0.9 or above | `"flooded"` |

(Text values go in quotes. You can rely on `elif`: when it is reached, the earlier conditions have already failed, so you do not need to repeat them.)

In [ ]:
fw = 0.62

# YOUR CODE HERE
fw_class = None

In [ ]:
# Check (do not edit)
assert fw_class == "high", f"fw = 0.62 should be 'high', got {fw_class!r}"
print("Correct. Try fw = 0.05, 0.5, 0.9 and 0.95 too, and check the boundaries.")

<details><summary>Solution</summary>

```python
if fw < 0.1:
    fw_class = "dry"
elif fw < 0.5:
    fw_class = "moderate"
elif fw < 0.9:
    fw_class = "high"
else:
    fw_class = "flooded"
```
</details>

Notice the problem: if you now need to classify 200 wells, you would copy these lines 200 times. That is what functions are for.

## 4. Functions

### 4.1 Functions you already know

A **function** is a named piece of code. You *call* it by writing its name with brackets, giving it **arguments** (inputs). It gives back a **result**.

| Function | What it does | Example |
|---|---|---|
| `len(x)` | number of items | `len([4, 5, 6])` gives `3` |
| `abs(x)` | size of a number, without sign | `abs(-4.5)` gives `4.5` |
| `round(x, n)` | round to `n` decimals | `round(2.567, 1)` gives `2.6` |
| `max(a, b, ...)`, `min(...)` | largest / smallest | `max(3, 7)` gives `7` |
| `float(text)` | text to decimal number | `float("0.22")` gives `0.22` |
| `int(x)` | cut off the decimals | `int(3.9)` gives `3` (it does **not** round) |
| `np.exp(x)` | $e^x$ | `np.exp(1)` gives `2.718...` |

Run it and check each answer against the table:

In [ ]:
print(len([4, 5, 6]))
print(abs(-4.5))
print(round(2.567, 1))
print(max(3, 7), min(3, 7))
print(float("0.22") + 1)
print(int(3.9))

### 4.2 Writing your own function

```python
def function_name(parameter1, parameter2):
    # indented lines: the body
    result = ...
    return result
```

- `def` starts the definition. The line ends with a colon and the body is indented.
- The names in the brackets are **parameters**: placeholders for the inputs.
- `return` sends the result back to whoever called the function and ends the function.
- **Defining** a function does nothing by itself. The body runs only when you **call** the function.

Here is equation (2) as a function. A line inside quotes right below the `def` (a **docstring**) describes what the function does; it is good practice to always write one.

In [ ]:
def exact_bacteria(alpha, N0, t):
    """Number of bacteria at time t (hours): the exact solution N0 * exp(alpha * t)."""
    N = N0 * np.exp(alpha * t)
    return N

print(exact_bacteria(1.74, 100, 2.0))     # call with three arguments
print(exact_bacteria(alpha, N0, 0.5))     # call using our variables
print(exact_bacteria(1.0, 10, 0.0))       # at t = 0 we expect N0 = 10

Things to notice:

- The function is defined once and called three times with different arguments.
- Names *inside* the function (`N`, and the parameters) are **local**: they exist only while the function runs. Try the next cell to see what that means (remove the `#`, read the error, put it back).

In [ ]:
# print(N)

You get `NameError: name 'N' is not defined`. That is a *feature*: a function cannot accidentally overwrite your variables.

### 4.3 `print` is not `return`

This is a common confusion. `print` **shows** something on the screen. `return` **hands a value back** so the rest of the program can use it. Predict what the last two lines print, then run:

In [ ]:
def show_double(x):
    print(2 * x)          # shows the number, but returns nothing

def double(x):
    return 2 * x          # hands the number back

a = show_double(4)
b = double(4)
print("a is", a)
print("b is", b)

`show_double` printed 8, but `a` is `None` (Python's word for "nothing"). `b` holds 8 and can be used in further calculations, e.g. `double(4) + 1`. **When you want a result to use later, `return` it.**

### Your turn: three functions

**Your turn 4.1.** Write `rel_error_percent(approx, exact)`: the size of the error relative to the exact value, in percent, i.e. $|{\rm approx}-{\rm exact}|/|{\rm exact}|\times 100$.

In [ ]:
def rel_error_percent(approx, exact):
    """Relative error of approx compared with exact, in percent."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert rel_error_percent(90, 100) == 10, f"rel_error_percent(90, 100) should be 10, got {rel_error_percent(90, 100)}"
assert abs(rel_error_percent(110, 100) - 10) < 1e-9, "the error must be positive even when approx is above exact (use abs)"
assert abs(rel_error_percent(2813.08, 3245.97) - 13.33) < 0.01, "check the formula: divide by exact, multiply by 100"
print("Correct")

<details><summary>Solution</summary>

```python
def rel_error_percent(approx, exact):
    """Relative error of approx compared with exact, in percent."""
    return abs(approx - exact) / abs(exact) * 100
```
</details>

**Your turn 4.2.** Turn Your turn 3.1 into a function `well_rate(demand, limit)` that returns the rate a well will produce (the demand, but never more than the limit). Use `if` / `else` inside the function.

In [ ]:
def well_rate(demand, limit):
    """Rate (STB/day) the well produces: the demand, capped at the limit."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert well_rate(3200, 2500) == 2500, "demand above the limit should return the limit"
assert well_rate(1800, 2500) == 1800, "demand below the limit should return the demand"
assert well_rate(2500, 2500) == 2500, "demand equal to the limit should return 2500"
print("Correct")

<details><summary>Solution</summary>

```python
def well_rate(demand, limit):
    """Rate (STB/day) the well produces: the demand, capped at the limit."""
    if demand > limit:
        return limit
    else:
        return demand
```
(A function can have several `return` lines; the first one reached ends the function. The built-in `min(demand, limit)` does the same in one line.)
</details>

**Your turn 4.3.** Turn Your turn 3.2 into a function `water_cut_class(fw)` that returns `"dry"`, `"moderate"`, `"high"` or `"flooded"` using the same table.

In [ ]:
def water_cut_class(fw):
    """Class of a well from its water cut (0 to 1)."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
tests = {0.05: "dry", 0.1: "moderate", 0.3: "moderate", 0.5: "high", 0.7: "high", 0.9: "flooded", 0.95: "flooded"}
for fw_test, expected in tests.items():
    got = water_cut_class(fw_test)
    assert got == expected, f"water_cut_class({fw_test}) should be {expected!r}, got {got!r}"
print("Correct")
# the function can now be used for many wells with a loop:
for w in [0.02, 0.35, 0.8, 0.97]:
    print(w, "->", water_cut_class(w))

<details><summary>Solution</summary>

```python
def water_cut_class(fw):
    """Class of a well from its water cut (0 to 1)."""
    if fw < 0.1:
        return "dry"
    elif fw < 0.5:
        return "moderate"
    elif fw < 0.9:
        return "high"
    else:
        return "flooded"
```
</details>

(The loop at the bottom of the check cell uses `for w in [...]`: a `for` loop that takes each item of a list in turn. We use this again in Section 7.)

## 5. When input goes wrong: `try` and `except`

Real data is untidy. A porosity might arrive as the text `"0.22"`, `"22%"` or `"n/a"`. Converting with `float()` works for the first and **crashes** for the others:

In [ ]:
print(float("0.22"))
# print(float("22%"))      # remove the # to see: ValueError

A `try` / `except` block says: *try this; if it fails with this kind of error, do that instead of crashing.*

```python
try:
    phi = float(text)
except ValueError:
    phi = None
```

`None` means "no value". The `except ValueError:` part only catches that kind of error, so other mistakes (a typo in a variable name, say) still show up and are not hidden.

**Your turn 5.1.** Write `parse_porosity(text)`. It receives text, and returns the porosity as a float if the text is a number strictly between 0 and 1. Otherwise it returns `None` (not a number, or outside the valid range).

In [ ]:
def parse_porosity(text):
    """Convert text to a porosity (0 < phi < 1), or return None if it is not valid."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert parse_porosity("0.22") == 0.22, "parse_porosity('0.22') should be 0.22"
assert parse_porosity("abc") is None, "text that is not a number should give None"
assert parse_porosity("22%") is None, "'22%' is not a number: should give None"
assert parse_porosity("1.5") is None, "a porosity above 1 should give None"
assert parse_porosity("-0.1") is None, "a negative porosity should give None"
assert parse_porosity("0") is None, "0 is not strictly between 0 and 1"
print("Correct")

<details><summary>Solution</summary>

```python
def parse_porosity(text):
    """Convert text to a porosity (0 < phi < 1), or return None if it is not valid."""
    try:
        phi = float(text)
    except ValueError:
        return None
    if 0 < phi < 1:
        return phi
    else:
        return None
```
</details>

## 6. Wrapping the Day 1 model in a function

Now the main use of functions today. Here is the Day 1 loop again, as a function with parameters instead of fixed numbers. Read it and compare with Day 1 Section 6.

**Your turn 6.1.** Complete `euler_growth(alpha, N0, dt, t_end)`. It must return **two lists**, the times and the values of $N$ *including the starting point*, using equation (4): `N_new = (1 + alpha * dt) * N_old`. (`return times, values` returns both at once.)

In [ ]:
def euler_growth(alpha, N0, dt, t_end):
    """Numerical solution of dN/dt = alpha*N with time step dt. Returns (times, values)."""
    n_steps = round(t_end / dt)
    times = [0.0]
    values = [N0]
    # YOUR CODE HERE: a for loop with n in range(1, n_steps + 1)
    return times, values

In [ ]:
# Check (do not edit)
ts, ns = euler_growth(1.74, 100, 0.05, 2.0)
assert len(ts) == len(ns) == 41, f"Expected 41 points (the start plus 40 steps), got {len(ts)}"
assert ns[0] == 100, "The first value should be N0"
assert abs(ns[3] - 128.4366) < 0.001, f"ns[3] = {ns[3]:.4f}, expected 128.4366 (your hand calculation from Day 1)"
assert abs(ns[-1] - 2813.08) < 0.01, f"The last value is {ns[-1]:.2f}, expected 2813.08"
assert abs(ts[-1] - 2.0) < 1e-9, "The last time should be t_end"
print("Correct")

<details><summary>Solution</summary>

```python
def euler_growth(alpha, N0, dt, t_end):
    """Numerical solution of dN/dt = alpha*N with time step dt. Returns (times, values)."""
    n_steps = round(t_end / dt)
    times = [0.0]
    values = [N0]
    for n in range(1, n_steps + 1):
        N_new = (1 + alpha * dt) * values[-1]
        values.append(N_new)
        times.append(n * dt)
    return times, values
```
</details>

When a function returns two things, you can catch them in two variables at once:

```python
times, values = euler_growth(1.74, 100, 0.05, 2.0)
```
(Python is packing the two results into a *tuple* and unpacking it. We study tuples on Day 6.)

## 7. Which time step is small enough?

Now we can ask the question we started with. In Day 1 we changed `dt` by hand and re-ran cells. With a function we can loop over a **list of time steps**. A `for` loop can take its values straight from a list:

```python
for dt in [0.5, 0.1, 0.05]:
    ...          # runs three times, with dt = 0.5, then 0.1, then 0.05
```

Read this cell and then run it. It uses the functions you wrote: `euler_growth`, `exact_bacteria` and `rel_error_percent`.

In [ ]:
t_end = 2.0
exact_end = exact_bacteria(alpha, N0, t_end)

print(f"{'dt (h)':>8} {'steps':>7} {'N numerical':>13} {'error %':>9}")
for dt in [0.5, 0.1, 0.05, 0.01, 0.001]:
    times, values = euler_growth(alpha, N0, dt, t_end)
    error = rel_error_percent(values[-1], exact_end)
    print(f"{dt:>8} {len(times) - 1:>7} {values[-1]:>13.1f} {error:>9.2f}")

(`:>8` in an f-string means "right-align in a space 8 characters wide"; it is what lines the columns up.)

The same table as Day 1, with one line instead of re-running cells. Now suppose the project says: *"the error must not exceed 5 %."*

**Your turn 7.1.** Loop over the five time steps again. For each one, compute the error and then:

- add the error to a list `errors` (use `.append`),
- add the text `"OK"` to a list `verdicts` if the error is at most `tolerance` (5 percent), otherwise add `"too coarse"`.

After the loop, `errors` and `verdicts` each have five items, in the same order as the time steps.

In [ ]:
tolerance = 5.0        # percent
dt_list = [0.5, 0.1, 0.05, 0.01, 0.001]

errors = []
verdicts = []

# YOUR CODE HERE

In [ ]:
# Check (do not edit)
assert len(errors) == 5 and len(verdicts) == 5, f"Both lists need 5 items; errors has {len(errors)}, verdicts has {len(verdicts)}"
assert abs(errors[0] - 62.33) < 0.05, f"errors[0] = {errors[0]:.2f}, expected about 62.33 (dt = 0.5)"
assert abs(errors[-1] - 0.30) < 0.01, f"errors[-1] = {errors[-1]:.2f}, expected about 0.30 (dt = 0.001)"
assert verdicts == ["too coarse", "too coarse", "too coarse", "OK", "OK"], f"verdicts = {verdicts}"
print("Correct: the largest acceptable time step in the list is", dt_list[verdicts.index("OK")])

<details><summary>Solution</summary>

```python
for dt in dt_list:
    times, values = euler_growth(alpha, N0, dt, t_end)
    error = rel_error_percent(values[-1], exact_end)
    errors.append(error)
    if error <= tolerance:
        verdicts.append("OK")
    else:
        verdicts.append("too coarse")
```
</details>

You just did what a simulation engineer does all the time: *run the model for several settings and decide which is acceptable*. The check cell used `verdicts.index("OK")`, which finds the position of the first `"OK"` in the list; you can use it to explore your own results.

**Your turn 7.2 (open-ended).** Which value of `dt` would you choose for this model, and why? Think about accuracy against the number of steps (cost), and write one or two sentences as a comment in the next cell. There is no single right answer.

In [ ]:
# my choice and reason:

## 8. Bridge: decisions inside `kusim`

The simulator reads property tables (oil formation volume factor against pressure, relative permeability against saturation). The thesis (Chapter 5, section 15) states how a value is read from a table, in exactly the three cases an `if` / `elif` / `else` expresses:

1. if the requested value is **below the first** table value, return the first result,
2. if it is **above the last** table value, return the last result,
3. otherwise, interpolate in a straight line between the two surrounding rows.

As Python (the third branch is Day 5's job):

```python
if x < x_table[0]:
    y = y_table[0]               # case 1: [0] is the first item
elif x > x_table[-1]:
    y = y_table[-1]              # case 2: [-1] is the last item
else:
    ...                          # case 3: interpolate (Day 5)
```

You already know every piece: the comparisons, the `if`/`elif`/`else` structure, and the `[0]` / `[-1]` indexes from Day 1. In `src/kusim/tables.py` the library function `np.interp` does all three cases for us; on Day 5 you will write your own version and compare.

The same file also contains guard conditions like yours in `parse_porosity`: for example `Table.__init__` stops with an error if a table has fewer than two points, because interpolation is impossible then:

```python
if len(self.x) < 2:
    raise ValueError("a table needs at least two points")
```
`raise` is the opposite of `except`: it *creates* an error on purpose, with a message that says what is wrong.

## 9. Self-quiz

Write short answers as comments, then open the answers.

1. What is the difference between `=` and `==`?
2. `x = 7`. What do `x > 5 and x < 10`, `x > 5 and x > 10` and `not x > 5` give?
3. What does a function return if it has no `return` line? What does `print` inside a function do to its result?
4. In an `if` / `elif` / `elif` / `else` chain, if two conditions are true, how many branches run?
5. Why does `0.1 + 0.2 == 0.3` give `False`, and what would you write instead?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.

<details><summary>Answers</summary>

1. `=` stores a value in a variable. `==` asks whether two values are equal and gives `True` or `False`.
2. `True`, `False` and `False`.
3. It returns `None`. `print` only shows text on the screen; it does not change what the function returns.
4. Only one: the first one whose condition is true. The rest are skipped.
5. Decimals like 0.1 are stored with a tiny rounding error, so the sum is not exactly 0.3. Compare with a tolerance: `abs(a - b) < 1e-9`.
</details>

## 10. Pocket reference (today's Python)

| You want to... | Write |
|---|---|
| ask a question | `x > 5`, `x == 5`, `x != 5`, `x <= 5` |
| combine questions | `A and B`, `A or B`, `not A` |
| choose | `if cond:` / `elif cond:` / `else:` (indented blocks) |
| compare decimals | `abs(a - b) < 1e-9` |
| text to number | `float("0.22")`, `int("7")` |
| catch a bad conversion | `try:` ... `except ValueError:` ... |
| define a function | `def name(a, b):` (indented body) with `return result` |
| call a function | `name(1, 2)` |
| return two results | `return a, b`  then  `x, y = name(...)` |
| loop over a list | `for item in my_list:` |
| position of a value in a list | `my_list.index(value)` |

## 11. Wrap-up

Today you:

- asked yes/no questions with comparisons and combined them with `and`, `or`, `not`,
- made the program choose with `if` / `elif` / `else`,
- guarded against bad input with `try` / `except`,
- wrote five functions (`rel_error_percent`, `well_rate`, `water_cut_class`, `parse_porosity`, `euler_growth`) and learned the difference between `print` and `return`,
- used a function and a loop over a list to test five time steps and pick one against an error tolerance.

**Next, Day 3:** `while` loops (repeat *until* something happens), loop patterns such as counting, summing and finding a maximum, and the logistic model, where a time step that is too large makes the answer oscillate.